### 2. UNWTO / UN Tourism (vía Our World in Data)
- **Fuente**: `https://ourworldindata.org/grapher/international-tourist-trips.csv`
- **Granularidad**: anual, a nivel país (no por destino ni por mes)
- **Uso en el proyecto**: contexto macro — dimensiona el problema de overtourism a nivel nacional; variable de control en los modelos
- **Limitación**: no sustituye al INE para el análisis por destino/mes

In [18]:
import pandas as pd

# Datos de llegadas de turistas internacionales, fuente original: UN Tourism (UNWTO)
# republicados por Our World in Data en formato CSV descargable directamente
url = "https://ourworldindata.org/grapher/international-tourist-trips.csv?v=1&csvType=full&useColumnShortNames=false"

df_unwto = pd.read_csv(url, storage_options={"User-Agent": "Mozilla/5.0"})

print(df_unwto.shape)
print(df_unwto.head())
print(df_unwto.columns.tolist())

(5241, 5)
    Entity Code  Year  Arrivals of tourists from abroad  \
0  Albania  ALB  2007                         1062000.0   
1  Albania  ALB  2008                         1247000.0   
2  Albania  ALB  2009                         1711000.0   
3  Albania  ALB  2010                         2191000.0   
4  Albania  ALB  2011                         2469000.0   

  World region according to OWID  
0                         Europe  
1                         Europe  
2                         Europe  
3                         Europe  
4                         Europe  
['Entity', 'Code', 'Year', 'Arrivals of tourists from abroad', 'World region according to OWID']


Le damos la granularidad que necesitamos de unicamente España

In [21]:
df_espana = df_unwto[df_unwto["Entity"] == "Spain"].sort_values("Year")
print(df_espana[["Year", "Arrivals of tourists from abroad"]])

      Year  Arrivals of tourists from abroad
4487  1995                        32971000.0
4488  1996                        34027000.0
4489  1997                        39553000.0
4490  1998                        41892000.0
4491  1999                        45440000.0
4492  2000                        46403000.0
4493  2001                        48565000.0
4494  2002                        50331000.0
4495  2003                        50854000.0
4496  2004                        52430000.0
4497  2005                        55914000.0
4498  2006                        58004000.0
4499  2007                        58666000.0
4500  2008                        57192000.0
4501  2009                        52178000.0
4502  2010                        52677000.0
4503  2011                        56177000.0
4504  2012                        57464000.0
4505  2013                        60675000.0
4506  2014                        64939000.0
4507  2015                        68175000.0
4508  2016

### 1. INE — Encuesta de Ocupación Hotelera
- **Tabla**: `2078` (Tempus3) — "Viajeros y pernoctaciones por puntos turísticos"
- **Endpoint**: `https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/2078`
- **Granularidad**: mensual, por punto turístico (municipio/destino), desglosado por tipo de variable (Viajeros/Pernoctaciones) y origen del viajero (residente en España/Extranjero)
- **Uso en el proyecto**: variable principal para medir presión turística y estacionalidad por destino — base para comparar el hotspot vs. alternativas
- **Notas técnicas**:
  - Requiere `User-Agent` en la petición (el default de `requests` es bloqueado)
  - El campo `Fecha` viene con desfase de timezone — usar `Anyo` + `FK_Periodo` para construir la fecha real
  - El `Nombre` de cada serie sigue el patrón `{Destino}. {Variable}. Total categorías. Residentes en {tipo}. Dato.`

#### Descarga de datos — Tablas INE (Ocupación en alojamientos turísticos)

| Tabla | Descripción | Granularidad |
|-------|-------------|--------------|
| `48423` | Número de alojamientos turísticos y noches ocupados por residencia del viajero | Nacional y comunidades autónomas |
| `48424` | Número de alojamientos turísticos y noches ocupados por residencia del viajero. | Puntos turísticos |
| `48425` | Pernoctaciones por residencia del viajero|Nacional y comunidades autónomas |
| `48426` | Pernoctaciones por residencia del viajero. | Puntos turísticos |

In [35]:
import requests

def descargar_tabla_ine(tabla_id, nult=24):
    """
    Descarga el JSON crudo de una tabla del INE (formato Tempus3).
    
    Parámetros:
    - tabla_id: número de la tabla (ej. 2078, 48423)
    - nult: cuántos últimos periodos traer
    """
    headers = {"User-Agent": "Mozilla/5.0"}  # el INE bloquea el user-agent por defecto
    url = f"https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/{tabla_id}"
    params = {"nult": nult}
    
    resp = requests.get(url, params=params, headers=headers)
    resp.raise_for_status()
    return resp.json()


# Descargamos las 3 tablas y guardamos el JSON crudo de cada una
tablas_ids = [48423, 48424, 48425,48426]
datos_crudos = {}

for tabla_id in tablas_ids:
    print(f"Descargando tabla {tabla_id}...")
    datos_crudos[tabla_id] = descargar_tabla_ine(tabla_id)
    print(f"  -> {len(datos_crudos[tabla_id])} series recibidas\n")

Descargando tabla 48423...
  -> 180 series recibidas

Descargando tabla 48424...
  -> 351 series recibidas

Descargando tabla 48425...
  -> 60 series recibidas

Descargando tabla 48426...
  -> 117 series recibidas



In [37]:
import pandas as pd

def json_a_dataframe(data):
    """
    Convierte la respuesta JSON de una tabla del INE (lista de series) 
    en un DataFrame tabular, con una fila por serie x periodo.
    """
    filas = []
    for serie in data:
        nombre = serie["Nombre"]
        for punto in serie["Data"]:
            filas.append({
                "Nombre": nombre,
                "Anyo": punto["Anyo"],
                "FK_Periodo": punto["FK_Periodo"],
                "Valor": punto["Valor"],
                "Secreto": punto["Secreto"],
            })
    
    df = pd.DataFrame(filas)
    
    # Reconstruimos la fecha real a partir de Anyo + FK_Periodo
    # (el campo "Fecha" original tiene problema de timezone)
    df["fecha_real"] = pd.to_datetime(
        df["Anyo"].astype(str) + "-" + df["FK_Periodo"].astype(str) + "-01"
    )
    
    return df.sort_values(["Nombre", "fecha_real"]).reset_index(drop=True)


# Aplanamos las 4 tablas ya descargadas (usando el JSON crudo guardado en datos_crudos)
dataframes = {}

for tabla_id, data in datos_crudos.items():
    dataframes[tabla_id] = json_a_dataframe(data)
    print(f"Tabla {tabla_id}: {dataframes[tabla_id].shape[0]} filas, {dataframes[tabla_id]['Nombre'].nunique()} series únicas")

# Acceso individual:
df_48423 = dataframes[48423]
df_48424 = dataframes[48424]
df_48425 = dataframes[48425]
df_48426 = dataframes[48426]

Tabla 48423: 3807 filas, 180 series únicas
Tabla 48424: 7398 filas, 351 series únicas
Tabla 48425: 1269 filas, 60 series únicas
Tabla 48426: 2466 filas, 117 series únicas
